### TimeWeightedVectorStoreRetriever

In [17]:
!uv --version

uv 0.12.17 (635500036 2026-09-18 x86_64-pc-windows-msvc)


In [18]:
from dotenv import load_dotenv

load_dotenv()

True

In [19]:
from langchain_teddynote import logging

logging.langsmith("test0922")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0922


In [20]:
from datetime import datetime, timedelta

import faiss
from langchain_classic.docstore import InMemoryDocstore
from langchain_classic.retrievers import TimeWeightedVectorStoreRetriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model, index, InMemoryDocstore({}), {})

retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore, decay_rate=0.0000000000000000000000001, k=1
)

In [21]:
yesterday = datetime.now() - timedelta(days=1)

retriever.add_documents(
    [
        Document(
            page_content="테디노트 구독해 주세요.",
            metadata={"last_accessed_at": yesterday},
        )
    ]
)

retriever.add_documents([Document(page_content="테디노트 구독 하실건가요?Please!")])

['9c3ecf3d-b0ef-405b-b4ce-943e0d8b7e67']

In [22]:
retriever.invoke("테디노트")

[Document(metadata={'last_accessed_at': datetime.datetime(2026, 9, 22, 15, 7, 44, 834437), 'created_at': datetime.datetime(2026, 9, 22, 15, 7, 44, 315700), 'buffer_idx': 0}, page_content='테디노트 구독해 주세요.')]

In [23]:
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model, index, InMemoryDocstore({}), {})

retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore, decay_rate=0.999, k=1
)

In [24]:
yesterday = datetime.now() - timedelta(days=1)

retriever.add_documents(
    [
        Document(
            page_content="테디노트 구독해 주세요.",
            metadata={"last_accessed_at": yesterday},
        )
    ]
)

retriever.add_documents([Document(page_content="테디노트 구독 해주실꺼죠? Please!")])

['d52edfe2-d156-468e-98ef-b9d1807fa23f']

In [25]:
retriever.invoke("테디")

[Document(metadata={'last_accessed_at': datetime.datetime(2026, 9, 22, 15, 7, 45, 388298), 'created_at': datetime.datetime(2026, 9, 22, 15, 7, 45, 46415), 'buffer_idx': 1}, page_content='테디노트 구독 해주실꺼죠? Please!')]

In [26]:
import datetime

from langchain_core.utils import mock_now

with mock_now(datetime.datetime(2024, 8, 30, 00, 00)):
    print(datetime.datetime.now())

2024-08-30 00:00:00


In [31]:
import datetime
from langchain_core.utils import mock_now

# 점수 계산에 실제로 쓰이는 건 retriever.memory_stream이므로 여기를 수정해야 함
for doc in retriever.memory_stream:
    doc.metadata["last_accessed_at"] = datetime.datetime(2024, 8, 30, 0, 0)
    doc.metadata["created_at"] = datetime.datetime(2024, 8, 30, 0, 0)

with mock_now(datetime.datetime(2024, 8, 30, 0, 0)):
    print(datetime.datetime.now())

    with mock_now(datetime.datetime(2024, 8, 29, 0, 0)):
        print(retriever.invoke("테디"))

2024-08-30 00:00:00
[Document(metadata={'last_accessed_at': MockDateTime(2024, 8, 29, 0, 0), 'created_at': datetime.datetime(2024, 8, 30, 0, 0), 'buffer_idx': 1}, page_content='테디노트 구독 해주실꺼죠? Please!')]


d:\hanhwa0902\ex0922\.venv\Lib\site-packages\langchain_classic\retrievers\time_weighted_retriever.py:80: RuntimeWarning: overflow encountered in cast
  score += vector_relevance
